In [10]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import fiona
import os
import io
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.functions import read_census_table, read_wp23_table

In [11]:
#TN we're reading in from UrbanSIM
tnurbansim = pd.read_csv(r"G:\3-GitHub\FCLUCS_Scenario_Planning\1-data\1-raw\2050RTP-TDMFeedback_2050Block.csv")

In [12]:
col_dict = {'sector_0': 'EMP_OTHER', 'sector_1': 'EMP_SERVICE', 'sector_2': 'EMP_RETAIL', 'sector_3': 'EMP_OFFICE', 'geo_level_id': 'GEO_ID',
               'sector_4': 'EMP_MEDICAL', 'sector_5': 'EMP_INDUSTRIAL', 'sector_6': 'EMP_GOVERNMENT', 'sector_7': 
               'EMP_FOODSERVICE', 'sector_8': 'EMP_EDUCATION', 'household_population': 'HHPOP', 'sum_total_households': 'HH', 'sum_total_jobs': 'EMP'}
tnurbansim = tnurbansim.rename(columns = col_dict)
tnurbansim_filt = tnurbansim[['GEO_ID', 'HHPOP', 'EMP', 'EMP_OTHER', 'EMP_SERVICE', 'EMP_RETAIL', 'EMP_OFFICE', 
'EMP_MEDICAL', 'EMP_INDUSTRIAL', 'EMP_GOVERNMENT', 'EMP_FOODSERVICE', 'EMP_EDUCATION']]

In [40]:
#notably, these aren't rounded to whole ppl. Since we're hexing this I don't think it matters.
tnurbansim_filt.head(2)

,GEO_ID,HHPOP,EMP,EMP_OTHER,EMP_SERVICE,EMP_RETAIL,EMP_OFFICE,EMP_MEDICAL,EMP_INDUSTRIAL,EMP_GOVERNMENT,EMP_FOODSERVICE,EMP_EDUCATION
0,470831203003000,16.582708,2.0,1.0,0.0,0.0,0.0,0.036140,0.035221,0.498547,0.286435,0.030014
1,470831203003090,0.231629,2.0,1.0,0.0,0.0,0.0,0.058606,0.049506,0.517907,0.301769,0.029002


In [14]:
tnurbansim_filt['GEO_ID'] = tnurbansim_filt['GEO_ID'].astype(str)

In [15]:
#KY we will take the same allocations and do 2050 controls, so read in the base year with allocations as well as the WP smoothed totals
ky = gpd.read_file(r"G:\3-GitHub\FCLUCS_Scenario_Planning\1-data\2-interim\BaseYear.gpkg", layer="KY_HHPOP")
kycontrols = read_wp23_table('DemSmoothed')

In [16]:
kycontrols.head(2)

,NAME,Variable,2023 Census,2023 WP,2024,2025,2026,2027,2028,2029,...,2042,2043,2044,2045,2046,2047,2048,2049,2050,Last Update Pushed_20260812
0,"Cannon County, Tennessee",Population,15063,14854.0,15122.259259,15180.518519,15239.777778,15300.037037,15359.296296,15419.555556,...,16232.925926,16297.185185,16362.444444,16427.703704,16492.962963,16559.222222,16625.481481,16691.740741,16759.0,None
1,"Cheatham County, Tennessee",Population,42254,42068.0,42485.111111,42718.222222,42952.333333,43188.444444,43424.555556,43663.666667,...,46893.111111,47152.222222,47412.333333,47673.444444,47936.555556,48201.666667,48467.777778,48735.888889,49005.0,None


In [17]:
thelist = ['Trigg County, Kentucky', 'Christian County, Kentucky']
kyhorizon = kycontrols[['NAME', '2023 Census', '2050']].loc[kycontrols['NAME'].isin(thelist)]
kyhorizon['HHPOP_GROWTH'] = kyhorizon['2050'] - kyhorizon['2023 Census']

In [18]:
kyhorizon.head(2)

,NAME,2023 Census,2050,HHPOP_GROWTH
17,"Christian County, Kentucky",72032,70260.0,-1772.0
19,"Trigg County, Kentucky",14369,16279.0,1910.0


In [19]:
trigghorizoncontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Trigg County, Kentucky', '2050'].iloc[0])
christianhorizoncontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Christian County, Kentucky', '2050'].iloc[0])
triggcontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Trigg County, Kentucky', '2023 Census'].iloc[0])
christiancontrol = int(kyhorizon.loc[kyhorizon['NAME'] == 'Christian County, Kentucky', '2023 Census'].iloc[0])
triggdiff = int(kyhorizon.loc[kyhorizon['NAME'] == 'Trigg County, Kentucky', 'HHPOP_GROWTH'].iloc[0])
christiandiff = int(kyhorizon.loc[kyhorizon['NAME'] == 'Christian County, Kentucky', 'HHPOP_GROWTH'].iloc[0])

In [20]:
controls = {'Trigg County, Kentucky': trigghorizoncontrol,'Christian County, Kentucky': christianhorizoncontrol}

ky['HHPOP_2050'] = ((ky['HHPOP_ALLOC'])/100) * ky['COUNTY'].map(controls)

In [21]:
ky.loc[ky['COUNTY'] == 'Trigg County, Kentucky', 'HHPOP_2050'].sum()

np.float64(16279.0)

In [34]:
ky = ky.copy()

# Save original 2050 allocation
ky["HHPOP_2050_ORIG"] = ky["HHPOP_2050"].copy()

# Calculate area in square miles using a projected CRS
ky_proj = ky.to_crs("EPSG:5070")

ky["AREA_SQMI"] = ky_proj.geometry.area / 2_589_988.11

# Calculate household population density
ky["HHPOP_DENSITY"] = (
    ky["HHPOP"] / ky["AREA_SQMI"].replace(0, np.nan)
)

# Identify zero-population blocks
ky["ZERO_POP"] = ky["HHPOP"] == 0

In [35]:
ky["HIGH_DENSITY"] = False

for county, group in ky.groupby("COUNTYFP"):

    populated = group.loc[
        (group["HHPOP"] > 0) &
        (group["HHPOP_DENSITY"].notna())
    ]

    if populated.empty:
        continue

    threshold = populated["HHPOP_DENSITY"].quantile(0.75)

    ky.loc[
        populated.index,
        "HIGH_DENSITY"
    ] = populated["HHPOP_DENSITY"] >= threshold

In [36]:
ky["_block_id"] = np.arange(len(ky))

zero_blocks = ky.loc[ky["ZERO_POP"]].copy()
high_blocks = ky.loc[ky["HIGH_DENSITY"]].copy()

adjacent = gpd.sjoin(
    zero_blocks,
    high_blocks[
        ["_block_id", "COUNTYFP", "HHPOP", "HHPOP_DENSITY", "geometry"]
    ],
    how="inner",
    predicate="touches",
    lsuffix="zero",
    rsuffix="high"
)

# Restrict adjacency to the same county
adjacent = adjacent.loc[
    adjacent["COUNTYFP_zero"] == adjacent["COUNTYFP_high"]
].copy()

In [37]:
weights = (
    adjacent.groupby("_block_id_zero")["HHPOP_high"]
    .sum()
    .rename("ADJ_POP")
)

ky["ADJ_POP"] = ky["_block_id"].map(weights).fillna(0)

ky["ELIGIBLE"] = (
    ky["ZERO_POP"] &
    (ky["ADJ_POP"] > 0)
)

In [38]:
ky["HHPOP_2050_FINAL"] = ky["HHPOP_2050_ORIG"].copy()

for county, idx in ky.groupby("COUNTYFP").groups.items():

    # County totals
    pop_2020 = ky.loc[idx, "HHPOP"].sum()
    pop_2050 = ky.loc[idx, "HHPOP_2050_ORIG"].sum()

    # Projected population growth
    growth = pop_2050 - pop_2020

    # Only redistribute positive growth
    if growth <= 0:
        continue

    # Identify eligible zero-population blocks
    eligible_idx = ky.loc[idx].index[
        ky.loc[idx, "ELIGIBLE"]
    ]

    if len(eligible_idx) == 0:
        continue

    # 5% of projected growth
    growth_to_redistribute = growth * 0.05

    # Reduce existing 2050 allocations proportionally
    existing_idx = ky.loc[idx].index[
        ky.loc[idx, "HHPOP"] > 0
    ]

    existing_total = ky.loc[
        existing_idx, "HHPOP_2050_ORIG"
    ].sum()

    ky.loc[existing_idx, "HHPOP_2050_FINAL"] -= (
        ky.loc[existing_idx, "HHPOP_2050_ORIG"]
        / existing_total
        * growth_to_redistribute
    )

    # Allocate growth to eligible zero-population blocks
    weights = ky.loc[eligible_idx, "ADJ_POP"]

    ky.loc[eligible_idx, "HHPOP_2050_FINAL"] += (
        weights / weights.sum()
        * growth_to_redistribute
    )

In [39]:
qa = ky.groupby("COUNTY").agg(
    Pop_2020=("HHPOP", "sum"),
    Pop_2050_Original=("HHPOP_2050_ORIG", "sum"),
    Pop_2050_Final=("HHPOP_2050_FINAL", "sum"),
    Eligible_Blocks=("ELIGIBLE", "sum")
)

qa["Growth"] = qa["Pop_2050_Original"] - qa["Pop_2020"]

qa["Control_Diff"] = (
    qa["Pop_2050_Final"] - qa["Pop_2050_Original"]
)

print(qa.round(2))

                            Pop_2020  Pop_2050_Original  Pop_2050_Final  \
COUNTY                                                                    
Christian County, Kentucky     67114            70260.0         70260.0   
Trigg County, Kentucky         14311            16279.0         16279.0   

                            Eligible_Blocks  Growth  Control_Diff  
COUNTY                                                             
Christian County, Kentucky               95  3146.0           0.0  
Trigg County, Kentucky                   30  1968.0           0.0  
